In [0]:
%run  ../utils/utils

In [0]:
# 1. Ver o que existe fisicamente no squad3
print("===== ESTRUTURA FÍSICA DO SQUAD3 =====")
container_squad3 = service_client.get_file_system_client("squad3")
listar_pastas_squad3(container_squad3, recursive=False)

In [0]:
# 2. Comparar schema das tabelas Silver relevantes entre squad1 e squad3
print("\n===== COMPARAÇÃO DE SCHEMA: ecommerce_pedidos =====")
 
try:
    dt_squad1 = DeltaTable(get_delta_path("silver", "ecommerce_pedidos", STORAGE_OPTIONS), storage_options=STORAGE_OPTIONS)
    schema_squad1 = set(dt_squad1.schema().json()["fields"][i]["name"] for i in range(len(dt_squad1.schema().json()["fields"])))
    print("Colunas squad1:", sorted(schema_squad1))
except Exception as e:
    print("Erro ao ler squad1/silver/ecommerce_pedidos:", e)
    schema_squad1 = set()
 
try:
    dt_squad3 = DeltaTable(get_delta_path_squad3("silver", "ecommerce_pedidos", STORAGE_OPTIONS), storage_options=STORAGE_OPTIONS)
    schema_squad3 = set(dt_squad3.schema().json()["fields"][i]["name"] for i in range(len(dt_squad3.schema().json()["fields"])))
    print("Colunas squad3:", sorted(schema_squad3))
except Exception as e:
    print("Erro ao ler squad3/silver/ecommerce_pedidos:", e)
    schema_squad3 = set()
 
if schema_squad1 and schema_squad3:
    so_squad1 = schema_squad1 - schema_squad3
    so_squad3 = schema_squad3 - schema_squad1
    print("\nColunas só no squad1:", so_squad1 if so_squad1 else "(nenhuma)")
    print("Colunas só no squad3:", so_squad3 if so_squad3 else "(nenhuma)")
    print("Schemas idênticos?", schema_squad1 == schema_squad3)

In [0]:
# 3. Repita para as outras tabelas relevantes ao modelo de anomalia
for tabela in ["ecommerce_itens_pedido", "ecommerce_produtos", "ecommerce_clientes"]:
    print(f"\n===== COMPARAÇÃO DE SCHEMA: {tabela} =====")
    try:
        dt1 = DeltaTable(get_delta_path("silver", tabela, STORAGE_OPTIONS), storage_options=STORAGE_OPTIONS)
        cols1 = set(f["name"] for f in dt1.schema().json()["fields"])
        print("squad1:", sorted(cols1))
    except Exception as e:
        print(f"squad1/{tabela} não encontrada ou erro:", e)
        cols1 = set()
 
    try:
        dt2 = DeltaTable(get_delta_path_squad3("silver", tabela, STORAGE_OPTIONS), storage_options=STORAGE_OPTIONS)
        cols2 = set(f["name"] for f in dt2.schema().json()["fields"])
        print("squad3:", sorted(cols2))
    except Exception as e:
        print(f"squad3/{tabela} não encontrada ou erro:", e)
        cols2 = set()
 
    if cols1 and cols2:
        print("Idênticos?", cols1 == cols2, "| Diferença:", cols1.symmetric_difference(cols2))
 